In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df=pd.read_csv("../data/Sleep_health_and_lifestyle_dataset.csv",keep_default_na=False)
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
sns.countplot(x='Sleep Disorder',data=df)
plt.savefig("../figures/countplot_sleep_disorder.png",dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
sns.boxplot(x='Sleep Disorder',y='Sleep Duration',data=df)
plt.savefig("../figures/sleep_Durations.png",dpi=150,bbox_inches='tight')

In [ ]:
sns.boxplot(x='Sleep Disorder',y='Stress Level',data=df)
plt.savefig("../figures/stress_levels.png",dpi=150,bbox_inches='tight')

In [ ]:
num_df=df.select_dtypes(include ='number')
num_df.corr()
sns.heatmap(num_df.corr(),annot=True,cmap='coolwarm')
plt.savefig("../figures/correlation_heatmap.png",dpi=150,bbox_inches='tight')

In [ ]:
df=df.drop(columns=["Person ID"])
df[["Systolic_BP", "Diastolic_BP"]]=df["Blood Pressure"].str.split("/",expand=True).astype(int)
df=df.drop(columns=["Blood Pressure"])
df["BMI Category"]=df["BMI Category"].replace({"Normal Weight": "Normal"})
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
x=df.drop(columns=["Sleep Disorder"])
y=df["Sleep Disorder"]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y,)



In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
num_col=x_train.select_dtypes(include='number').columns.tolist()
cat_col=x_train.select_dtypes(include='object').columns.tolist()
preprocessor=ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),num_col),
        ("cat",OneHotEncoder(handle_unknown="ignore"),cat_col)
    ]
)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyClassifier

models={
    "Logistic Regression":LogisticRegression(max_iter=1000),
    "Decision Tree":DecisionTreeClassifier(random_state=42),
    "Random Forest":RandomForestClassifier(random_state=42)
}
results=[]
baseline=DummyClassifier(strategy='most_frequent')
baseline.fit(x_train,y_train)

for name,cla in models.items():
    pipe=Pipeline(steps=[
        ("preprocessor",preprocessor),
        ("classifier",cla)
    ])
    cv=cross_val_score(pipe,x_train,y_train,cv=5)
    pipe.fit(x_train,y_train)
    results.append({
        "name":name,
        "mean_cv_score":cv.mean().round(3),
        "cv_std":cv.std().round(3),
        "train_score":pipe.score(x_train,y_train)
    })
results_df=pd.DataFrame(results)
print("Baseline score:",baseline.score(x_train,y_train))
results_df



In [ ]:
from sklearn.metrics import classification_report,ConfusionMatrixDisplay
best=Pipeline([("preprocessor",preprocessor),("classifier",LogisticRegression(max_iter=1000))])
best.fit(x_train,y_train)
y_pred=best.predict(x_test)
print(classification_report(y_test,y_pred))
ConfusionMatrixDisplay.from_predictions(y_test,y_pred)
plt.savefig("../figures/confusion_matrix.png",dpi=150,bbox_inches='tight')
plt.show()


In [ ]:
from sklearn.linear_model import LogisticRegression
lr=Pipeline([("preprocessor",preprocessor),("classifier",LogisticRegression(max_iter=1000))])
lr.fit(x_train,y_train)
feature_names=lr.named_steps["preprocessor"].get_feature_names_out()
coef=pd.DataFrame(
    lr.named_steps["classifier"].coef_,
    columns=feature_names,
    index=lr.named_steps["classifier"].classes_
).T
coef["strengths"]=coef.abs().max(axis=1)
top=coef.sort_values("strengths",ascending=False).head(8).drop(columns=["strengths"])
print(top)
top.plot(kind="barh",title="Top 8 feature importance(Logistic Regression)")
plt.xlabel("coefficient(Scaled feature)")
plt.savefig("../figures/top_8_feature_importance.png",dpi=150,bbox_inches='tight')
plt.show()

